# FTS Parameter Sweep & Out-of-Sample Backtest Evaluation

This notebook evaluates hyperparameter and architectural variations under **controlled isolation** (*ceteris paribus*) for any selected parameter sweep specification (`SweepSpec`).

### Core Principles:
1. **Fixed Baseline Hyperparameters:** All non-swept hyperparameters (learning rate, batch size, dropout, execution fees, slippage) are held strictly constant.
2. **Out-of-Sample Evaluation:** Models are trained on the training split, registered as candidate ONNX models in `ModelRegistryLog`, and backtested on an independent holdout test split using the `BacktestEngine`.
3. **Generalized Sweep Results & Visualizations:** Results are encapsulated in `SweepResult`, saved to JSON summary files, and visualized using interactive Plotly charts (`SweepVisualizer`) and standalone HTML reports (`HTMLSweepExporter`).

### 1. Import Dependencies & Set Pathing

In [1]:
import os
import sys
import logging

# Ensure src and project modules are on path
sys.path.insert(0, os.path.abspath("../src"))

from trading_bot.config import settings
from trading_bot.backtesting import SweepVisualizer, HTMLSweepExporter, SweepResult
from plugins.nets.spec import SweepSpec
from plugins.nets.training.sweep_runner import run_parameter_sweep

# Set logging level
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

### 2. Load Sweep Specification & Execute Controlled Sweep

Set `spec_path` to point to any parameter sweep specification YAML file. The sweep runner dynamically inspects `spec.sweep_param` and `spec.sweep_values`.

In [2]:
CHOSEN_MARKET = "BTCUSDT"
CHOSEN_MODEL = "xgboost"
SWEEP_PARAM = "max_depth"

spec_path = f"./specs/train/{CHOSEN_MARKET}/sweep/{CHOSEN_MODEL}_{SWEEP_PARAM}.yaml"
spec = SweepSpec.from_yaml(spec_path)

print(f"Loaded Sweep Spec     : '{spec.sweep_name}'")
print(f"Model Architecture    : {spec.model_type.upper()}")
print(f"Target Market         : {spec.market.market_id} ({spec.market.interval})")
print(f"Sweeping Parameter    : '{spec.sweep_param}' over {spec.sweep_values}")
print(f"Train Date Range      : {spec.train_dates.start_date} to {spec.train_dates.end_date}")
print(f"Test Date Range       : {spec.test_dates.start_date} to {spec.test_dates.end_date}")

sweep_result: SweepResult = run_parameter_sweep(spec)

print(f"\n=================== OUT-OF-SAMPLE SWEEP RESULTS ({sweep_result.sweep_param}) ===================")
print(f"Total Trials Completed: {len(sweep_result.trials)}")
print(f"Created At: {sweep_result.created_at}")

2026-07-03 15:48:52,452 - INFO - --- Running Sweep Trial [1/4]: max_depth = 2 ---


Loaded Sweep Spec     : 'xgboost_max_depth_sweep'
Model Architecture    : XGBOOST
Target Market         : BTC/USDT (30m)
Sweeping Parameter    : 'max_depth' over [2, 3, 5, 8]
Train Date Range      : 2025-11-04 19:30:00+00:00 to 2026-05-30 19:30:00+00:00
Test Date Range       : 2026-06-01 03:30:00+00:00 to 2026-06-21 23:00:00+00:00


2026-07-03 15:48:52,840 - INFO - Training XGBoost on 9937 bars.
2026-07-03 15:48:52,989 - INFO - XGBoost Validation: Best Iteration = 12, Loss = 0.000006, IC = 0.0323, Dir Acc = 0.4892
2026-07-03 15:48:53,016 - INFO - Model '0cf4b3f374cc' already registered in model_registry. Returning existing entry.
2026-07-03 15:48:53,018 - INFO - Model 0cf4b3f374cc (xgboost) successfully trained & registered.
2026-07-03 15:48:53,038 - INFO - Successfully deserialized FeaturePipeline from ONNX metadata.
2026-07-03 15:48:53,039 - INFO - Loaded ONNX model from /home/alfred/github/fts/models/registry/trials/0cf4b3f374cc.onnx
2026-07-03 15:48:53,039 - INFO - StrategyEngine initialized with 1 strategies: [nets_strategy_xgboost]
2026-07-03 15:48:53,039 - INFO - Portfolio initialized with cash: 10000.00 USD
2026-07-03 15:48:53,040 - INFO - FixedPercentageSizer initialized with percentage: 10.00%
2026-07-03 15:48:53,040 - INFO - RiskManager initialized with sizer: fixed_percentage, max_allocation: 25.0%, ma


=================== OUT-OF-SAMPLE SWEEP RESULTS (max_depth) ===================
Total Trials Completed: 4
Created At: 2026-07-03T18:48:52.452039+00:00


### 3. Interactive Sensitivity & Performance Visualization

Render the multi-view interactive Plotly chart combining parameter sensitivity curves, drawdown profiles, overlaid equity curves, and metric summary table.

In [3]:
visualizer = SweepVisualizer()
fig = visualizer.render_charts(sweep_result)
fig.show()

### 4. Export Visualization Report

Export standalone interactive HTML report using `HTMLSweepExporter`.

In [4]:
exporter = HTMLSweepExporter()
report_path = exporter.export(sweep_result)
print(f"Interactive Sweep Visualization HTML saved to: {report_path}")

2026-07-03 15:49:30,872 - INFO - Saving interactive sweep visualization HTML to: /home/alfred/github/fts/runs/reports/sweep_report_xgboost_max_depth_sweep.html


Interactive Sweep Visualization HTML saved to: /home/alfred/github/fts/runs/reports/sweep_report_xgboost_max_depth_sweep.html
